# MaterialMind-ECE — Phase 5: PCA & 3D Material Visualization
### Unit 3: Machine Learning / AI | Project 7: Electronic Material Clustering

This notebook implements **Phase 5: Principal Component Analysis (PCA) and High-Dimensional Visualization** on the 1,056 inorganic electronic materials using the standardized 6-feature matrix.

#### Core Objectives:
1. **Load clustered material data** (`data/processed/materials_clustered.csv`), scaler (`models/final_scaler.joblib`), and K-Means model (`models/kmeans.joblib`).
2. **Fit PCA** across all 6 standardized physical dimensions.
3. **Compute and analyze explained variance ratios, eigenvalues, and scree curves**.
4. **Interpret principal component loadings** based strictly on mathematical weights.
5. **Generate 2D and 3D visual projections** colored by K-Means cluster labels ($K=4$).
6. **Export low-dimensional coordinates** to `data/processed/pca_coordinates.csv`.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import plotly.graph_objects as go

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
%matplotlib inline
print("PCA environment initialized.")

## 1. Load Data, Scaler, and Trained K-Means Model

In [2]:
data_path = '../data/processed/materials_clustered.csv'
if not os.path.exists(data_path):
    data_path = 'data/processed/materials_clustered.csv'

scaler_path = '../models/final_scaler.joblib'
if not os.path.exists(scaler_path):
    scaler_path = 'models/final_scaler.joblib'

kmeans_path = '../models/kmeans.joblib'
if not os.path.exists(kmeans_path):
    kmeans_path = 'models/kmeans.joblib'

df = pd.read_csv(data_path)
scaler = joblib.load(scaler_path)
kmeans = joblib.load(kmeans_path)

features = ['band_gap', 'poly_total', 'poly_electronic', 'ionic_polarization_fraction', 'density', 'volume']
X_scaled = scaler.transform(df[features])

print(f"Loaded {df.shape[0]} materials across {len(features)} features.")
print(f"Cluster counts:\n{df['cluster'].value_counts().sort_index()}")

## 2. Apply Principal Component Analysis (PCA)
We compute all 6 principal components to capture the complete variance spectrum.

In [3]:
pca = PCA(n_components=6)
X_pca = pca.fit_transform(X_scaled)

evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

df_var = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(6)],
    'Eigenvalue': np.round(pca.explained_variance_, 4),
    'Explained Variance (%)': np.round(evr * 100, 2),
    'Cumulative Variance (%)': np.round(cum_evr * 100, 2)
})
df_var

### Scree Plot & Cumulative Variance Curve

In [4]:
fig, ax1 = plt.subplots(figsize=(8, 4.5))

bars = ax1.bar(range(1, 7), evr * 100, color='#1f78b4', alpha=0.75, label='Individual Explained Variance (%)')
ax2 = ax1.twinx()
line = ax2.plot(range(1, 7), cum_evr * 100, color='#e31a1c', marker='o', lw=2, label='Cumulative Variance (%)')

ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Individual Variance (%)', color='#1f78b4')
ax2.set_ylabel('Cumulative Variance (%)', color='#e31a1c')
ax1.set_xticks(range(1, 7))
ax1.set_xticklabels([f'PC{i}' for i in range(1, 7)])
ax2.set_ylim(20, 105)
ax2.axhline(74.66, color='#e31a1c', linestyle=':', label='Top-3 Cumulative (74.66%)')
plt.title('MaterialMind-ECE: PCA Scree Plot & Cumulative Variance', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Principal Component Loadings Matrix
We examine the mathematical weights (eigenvector coordinates) of each physical property across the principal components.

In [5]:
loadings = pd.DataFrame(pca.components_.T, index=features, columns=[f'PC{i+1}' for i in range(6)])

plt.figure(figsize=(9, 5))
sns.heatmap(loadings, annot=True, fmt='.3f', cmap='coolwarm', center=0, cbar_kws={'shrink': 0.8})
plt.title('Principal Component Feature Loadings (Weights)', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()
loadings.round(4)

### Interpretation of Principal Components:
1. **PC1 (37.08% Variance): Penn Model / Polarizability vs Bandgap**
   - Dominant Positive: `poly_electronic` (+0.567), `poly_total` (+0.521), `density` (+0.384)
   - Dominant Negative: `band_gap` (-0.440)
   - Represents the fundamental quantum-mechanical inverse relationship between energy bandgap and optical polarizability (Moss's Rule $\varepsilon_\infty \propto 1/E_g^2$).

2. **PC2 (21.00% Variance): Ionicity vs Structural Unit Cell Volume**
   - Dominant Positive: `ionic_polarization_fraction` (+0.604), `poly_total` (+0.387), `band_gap` (+0.373)
   - Dominant Negative: `volume` (-0.562)
   - Distinguishes compact, highly ionic polar dielectrics from large-volume, open-framework covalent crystals.

3. **PC3 (16.58% Variance): Cell Volume vs Mass Packing Density**
   - Dominant Positive: `volume` (+0.625), `poly_total` (+0.371)
   - Dominant Negative: `density` (-0.555)
   - Represents structural packing: large dilute unit cells versus compact, dense lattices composed of heavy elements.

## 4. 2D PCA Projection (PC1 vs PC2) with K-Means Clusters
We project all 1,056 materials into the PC1–PC2 plane and overlay the K-Means cluster centroids.

In [6]:
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]
df['PC3'] = X_pca[:, 2]

pca_centroids = pca.transform(kmeans.cluster_centers_)

cluster_colors = {0: '#2b83ba', 1: '#fdae61', 2: '#2ca02c', 3: '#d7191c'}
cluster_names = {
    0: 'Cluster 0: Dense / Moderate-Gap (N=412)',
    1: 'Cluster 1: Open-Framework / Large Volume (N=248)',
    2: 'Cluster 2: Wide-Gap / Ionic Dielectric (N=388)',
    3: 'Cluster 3: Colossal Permittivity / Narrow-Gap (N=8)'
}

plt.figure(figsize=(10, 7))
for c in range(4):
    sub = df[df['cluster'] == c]
    plt.scatter(sub['PC1'], sub['PC2'], c=cluster_colors[c], label=cluster_names[c],
                alpha=0.65 if c != 3 else 0.95, s=35 if c != 3 else 90,
                edgecolors='k' if c == 3 else 'none')

# Centroids
plt.scatter(pca_centroids[:, 0], pca_centroids[:, 1], c='black', marker='X', s=140, edgecolors='white', label='Centroids')
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.title('2D PCA Latent Projection (PC1 vs PC2)', fontweight='bold', fontsize=12)
plt.xlabel(f'PC1 ({evr[0]*100:.1f}% Variance): Polarizability vs Bandgap')
plt.ylabel(f'PC2 ({evr[1]*100:.1f}% Variance): Ionicity vs Volume')
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

## 5. Interactive 3D PCA Visualization (Plotly)
We render the 3D projection across (PC1, PC2, PC3) capturing **74.66%** of total dataset variance with full hover information.

In [7]:
df['Hover_Text'] = (
    "<b>Material ID:</b> " + df['material_id'] + "<br>" +
    "<b>Formula:</b> " + df['formula'] + "<br>" +
    "<b>Cluster:</b> " + df['cluster'].astype(str) + "<br>" +
    "<b>Band Gap:</b> " + df['band_gap'].round(2).astype(str) + " eV<br>" +
    "<b>Permittivity (ε_total):</b> " + df['poly_total'].round(2).astype(str) + "<br>" +
    "<b>Ionic Fraction (f_ionic):</b> " + df['ionic_polarization_fraction'].round(3).astype(str) + "<br>" +
    "<b>Density:</b> " + df['density'].round(2).astype(str) + " g/cm³<br>" +
    "<b>Volume:</b> " + df['volume'].round(1).astype(str) + " Å³"
)

fig3d = go.Figure()
plotly_colors = {0: 'rgb(43, 131, 186)', 1: 'rgb(253, 174, 97)', 2: 'rgb(44, 160, 44)', 3: 'rgb(215, 25, 28)'}

for c in range(4):
    sub = df[df['cluster'] == c]
    fig3d.add_trace(go.Scatter3d(
        x=sub['PC1'], y=sub['PC2'], z=sub['PC3'],
        mode='markers',
        name=cluster_names[c],
        marker=dict(size=4 if c != 3 else 7, color=plotly_colors[c], opacity=0.75 if c != 3 else 0.95),
        text=sub['Hover_Text'],
        hoverinfo='text'
    ))

fig3d.update_layout(
    title="<b>MaterialMind-ECE: 3D PCA Latent Space</b><br><sup>Total Cumulative Variance = 74.66% | N = 1,056 Materials</sup>",
    scene=dict(
        xaxis_title=f"PC1 ({evr[0]*100:.1f}%)",
        yaxis_title=f"PC2 ({evr[1]*100:.1f}%)",
        zaxis_title=f"PC3 ({evr[2]*100:.1f}%)"
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)
fig3d.show()

## 6. Export PCA Coordinates

In [8]:
processed_dir = '../data/processed' if os.path.exists('../data/processed') else 'data/processed'
pca_export_cols = ['material_id', 'formula', 'cluster', 'PC1', 'PC2', 'PC3']
out_csv = os.path.join(processed_dir, 'pca_coordinates.csv')
df_pca_all = pd.concat([df[['material_id', 'formula', 'cluster']], pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(6)])], axis=1)
df_pca_all.to_csv(out_csv, index=False)
print(f"Saved PCA coordinates to: {out_csv} ({df_pca_all.shape[0]} materials x {df_pca_all.shape[1]} columns)")